# Prometheus v1 — Colab Runtime Template

**Do not edit this notebook manually.** The harness layer may be updated by
the governor-approved Ashby loop; the gate and checkpoint cells are protected.

## How a session works
1. **Cell 1 – Config**: set `RUN_ID` and confirm the Drive path.
2. **Cell 2 – Mount & install**: mounts Drive, installs `prometheus-v1` from
   the pinned wheel on Drive, initialises the gate.
3. **Cell 3 – Gate check**: verifies the heartbeat; aborts if the governor
   has halted the run.
4. **Cell 4 – Run loop**: the actual episode loop. Checkpoints to Drive after
   every episode so a forced restart resumes cleanly.
5. **Cell 5 – Propose variant** *(optional)*: packages the current harness as
   a bundle and writes it to the proposals folder for governor review.

After a Colab forced restart, re-run from **Cell 2** (Drive stays mounted,
checkpoint is automatically resumed).

---
## Cell 1 — Configuration *(edit this cell)*

In [ ]:
# ── User-editable configuration ───────────────────────────────────────────────
RUN_ID = "run-001"          # must match the governor's policy run_id

# Google Drive paths (relative to MyDrive root)
DRIVE_ROOT      = "/content/drive/MyDrive"
PROMETHEUS_DIR  = f"{DRIVE_ROOT}/prometheus"          # gdrive:prometheus
EXCHANGE_DIR    = f"{PROMETHEUS_DIR}/{RUN_ID}"         # gdrive:prometheus/run-001
PUB_KEY_PATH    = f"{PROMETHEUS_DIR}/governor_public_key.txt"
WHEEL_GLOB      = f"{PROMETHEUS_DIR}/prometheus_v1-*.whl"  # optional pre-built wheel

# Local Colab paths
INSTALL_ROOT    = f"/content/prometheus_harness"       # where approved variants land
CHECKPOINT_PATH = f"{EXCHANGE_DIR}/checkpoint.json"   # survives runtime restarts

# Heartbeat tolerance (should match governor policy: heartbeat_ttl_s = 900)
MAX_CLOCK_SKEW_S = 60.0

print(f"RUN_ID        : {RUN_ID}")
print(f"Exchange dir  : {EXCHANGE_DIR}")
print(f"Public key    : {PUB_KEY_PATH}")
print(f"Install root  : {INSTALL_ROOT}")
print(f"Checkpoint    : {CHECKPOINT_PATH}")

---
## Cell 2 — Mount Drive & install package
*Re-run this after every forced restart.*

In [ ]:
import os, sys, json, time, glob, shutil, hashlib, subprocess, pathlib

# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("Drive mounted.")

# ── 2. Verify exchange directory and public key exist ─────────────────────────
for p in [EXCHANGE_DIR, PUB_KEY_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"Expected path not found: {p}\n"
            "Check that the governor has started and rclone has synced."
        )
print(f"Exchange dir  : OK  ({EXCHANGE_DIR})")

GOV_PUB_KEY_HEX = pathlib.Path(PUB_KEY_PATH).read_text().strip()
assert len(GOV_PUB_KEY_HEX) == 64, "governor_public_key.txt should be a 64-char hex string"
print(f"Governor key  : {GOV_PUB_KEY_HEX[:16]}…  (loaded)")

# ── 3. Install prometheus-v1 ──────────────────────────────────────────────────
# Prefer a pre-built wheel on Drive (faster, pinned version).
# Fall back to installing from PyPI / the repo if no wheel is found.
wheels = sorted(glob.glob(WHEEL_GLOB))
if wheels:
    wheel = wheels[-1]   # newest
    print(f"Installing from wheel: {os.path.basename(wheel)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", wheel])
else:
    print("No wheel found on Drive. Installing prometheus-v1 from repo (slow first run).")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "git+https://github.com/pmc/Prometheus_v1.git",  # replace with real URL when repo is public
    ])

# ── 4. Initialise the Gate ────────────────────────────────────────────────────
from prometheus.runtime.gate import Gate, Halted, InstallRefused

# The baseline hash is the generation-0 variant hash, read from the exchange.
# The governor writes it to baseline_hash.txt when it calls init_run.
baseline_hash_path = pathlib.Path(EXCHANGE_DIR) / "baseline_hash.txt"
if not baseline_hash_path.exists():
    raise FileNotFoundError(
        f"{baseline_hash_path} not found.\n"
        "Run `prometheus-governor init` on the workstation first."
    )
BASELINE_HASH = baseline_hash_path.read_text().strip()
print(f"Baseline hash : {BASELINE_HASH[:16]}…")

gate = Gate(
    governor_public_key_hex=GOV_PUB_KEY_HEX,
    run_id=RUN_ID,
    exchange=pathlib.Path(EXCHANGE_DIR),
    install_root=pathlib.Path(INSTALL_ROOT),
    baseline_hash=BASELINE_HASH,
    max_clock_skew_s=MAX_CLOCK_SKEW_S,
)
print(f"Gate ready    : generation={gate.generation}, "
      f"installed={gate.installed_hash[:16]}…")

---
## Cell 3 — Gate check
Run once after Cell 2. The run loop (Cell 4) calls this before every episode
automatically; this cell is here so you can test it manually.

In [ ]:
try:
    hb = gate.check_heartbeat()
    remaining = hb["expires_at"] - time.time()
    print(f"Heartbeat OK  : counter={hb['counter']}, "
          f"expires in {remaining:.0f}s, halt={hb.get('halt', False)}")
except Halted as e:
    print(f"HALTED: {e}")
    raise

---
## Cell 4 — Run loop
Runs Hidden-Law Lab episodes, checkpoints after each one.
Safe to interrupt and restart — it resumes from the checkpoint.

In [ ]:
from prometheus.lab import HiddenLawWorld
from prometheus.lab.laws import LAW_NAMES
from prometheus.runtime.sandbox import SandboxViolation, run as sandbox_run

# ── Checkpoint helpers ────────────────────────────────────────────────────────

def load_checkpoint() -> dict:
    """Load checkpoint from Drive, or return a fresh state."""
    p = pathlib.Path(CHECKPOINT_PATH)
    if p.exists():
        state = json.loads(p.read_text())
        print(f"Checkpoint resumed: episode {state['next_episode']}, "
              f"{len(state['results'])} results so far")
        return state
    print("No checkpoint found — starting fresh.")
    return {"next_episode": 0, "results": {}}


def save_checkpoint(state: dict) -> None:
    """Atomically write checkpoint to Drive."""
    p = pathlib.Path(CHECKPOINT_PATH)
    tmp = p.with_name(f".{p.name}.tmp")
    tmp.write_text(json.dumps(state, indent=2, sort_keys=True))
    tmp.replace(p)


# ── Episode configuration ─────────────────────────────────────────────────────

TOTAL_EPISODES = 200    # target for Phase 0 bake-off
NOISE          = 0.05
BUDGET         = 500.0

# Seeds: deterministic range so the governor can reproduce any world.
EPISODE_SEEDS  = list(range(TOTAL_EPISODES))

# ── Harness: run agent code inside the sandbox ────────────────────────────────
# Agent code is executed in a sandboxed subprocess with:
#   - no network access (socket/http/urllib blocked at import level)
#   - CPU time limit: 10 s (RLIMIT_CPU, kernel-enforced SIGKILL)
#   - Virtual memory limit: 256 MiB (RLIMIT_AS)
#   - Wall-clock timeout: 30 s
#   - No open(), eval(), exec(), subprocess, os, ctypes, …
#
# The agent receives `inputs` (a JSON-serialisable dict) and sets
# `return_value` to communicate its law claims back.
#
# Replace AGENT_CODE with the foundation-model output once the bake-off
# model is selected (Phase 0, step 5).

AGENT_CODE = """
# Placeholder harness: claims all laws are False.
# Replace with model-generated strategy code in Phase 0, step 5.
return_value = {name: False for name in inputs["law_names"]}
"""


def run_episode(world: HiddenLawWorld) -> dict[str, bool]:
    """Run AGENT_CODE in the sandbox and return law claims {law_name: bool}.

    On SandboxViolation (timeout, OOM, forbidden import) the episode is
    counted as a zero-score run rather than crashing the whole loop.
    """
    world_inputs = {
        "law_names": LAW_NAMES,
        "family":    world.family,
        "n_symbols": len(world.symbols),
        "budget":    world.budget,
        # Expose the observation API as pre-computed tables so the agent
        # doesn't need os/subprocess to interact with the world.
        # (Full observation tables will be added in Phase 0, step 5.)
    }
    try:
        result = sandbox_run(
            AGENT_CODE,
            inputs=world_inputs,
            cpu_seconds=10,
            wall_seconds=30,
            memory_mb=256,
        )
        if result.error:
            print(f"  [sandbox error] {result.error[:120]}")
            return {name: False for name in LAW_NAMES}
        claims = result.return_value
        if not isinstance(claims, dict):
            return {name: False for name in LAW_NAMES}
        return {name: bool(claims.get(name, False)) for name in LAW_NAMES}
    except SandboxViolation as exc:
        print(f"  [sandbox violation] {exc}")
        return {name: False for name in LAW_NAMES}


# ── Run loop ──────────────────────────────────────────────────────────────────

state = load_checkpoint()

for episode_idx in range(state["next_episode"], TOTAL_EPISODES):
    seed = EPISODE_SEEDS[episode_idx]

    # Gate check before every episode: halts if the governor pulled the kill switch.
    try:
        gate.check_heartbeat()
    except Halted as e:
        save_checkpoint(state)   # save progress before stopping
        print(f"\nRun halted by governor at episode {episode_idx}: {e}")
        raise SystemExit(0)      # clean exit; Colab will not auto-restart on SystemExit

    world = HiddenLawWorld(seed=seed, noise=NOISE, budget=BUDGET)
    claims = run_episode(world)
    truth  = world.ground_truth()
    correct = sum(claims.get(k) == v for k, v in truth.items())

    state["results"][str(seed)] = {
        "claims" : claims,
        "truth"  : truth,
        "correct": correct,
        "family" : world.family,
        "n"      : len(world.symbols),
        "spent"  : world.spent,
    }
    state["next_episode"] = episode_idx + 1
    save_checkpoint(state)

    if (episode_idx + 1) % 10 == 0 or episode_idx == TOTAL_EPISODES - 1:
        done   = state["next_episode"]
        avg    = sum(r["correct"] for r in state["results"].values()) / (done * len(LAW_NAMES))
        print(f"Episode {done:4d}/{TOTAL_EPISODES}  accuracy={avg:.3f}")

print("\nAll episodes complete.")
total  = len(state["results"])
avg    = sum(r["correct"] for r in state["results"].values()) / (total * len(LAW_NAMES))
print(f"Final accuracy: {avg:.3f}  ({total} worlds × {len(LAW_NAMES)} laws)")

---
## Cell 4b — Foundation-model bake-off *(Phase 0, step 5)*

Runs 200 Hidden-Law Lab episodes against the selected cortex model and logs
results to an MLflow file-store experiment on Drive.

**Prerequisites**
- vLLM is already serving the model on `localhost:8000` (see Cell 4b-setup).
- Drive is mounted and the gate is initialised (Cells 2–3).
- `pip install openai mlflow` has been run (Cell 2 does this automatically
  when the `colab` extra is installed).

**How to run a bake-off against all three candidates**

1. In Cell 4b-setup, start the vLLM server for the model you want.
2. Run Cell 4b-setup once.
3. Run Cell 4b-run once per model (change `MODEL_NAME` each time).
4. Run Cell 4b-compare to print the leaderboard.

The winner (highest `accuracy`) is the cortex model for Phase 0.
Pin its HF commit hash in `src/prometheus/bakeoff/models.py` → `revision`.

In [ ]:
# ── Cell 4b-setup: install extras and start vLLM ─────────────────────────────
# Run this cell once before each bake-off run.

import subprocess, sys

# Install runtime extras (openai client + mlflow)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "openai>=1.0", "mlflow>=2.0"])

# ── Optional: launch vLLM in the background ──────────────────────────────────
# Uncomment the block below to start vLLM serving your chosen model.
# This takes ~3–5 minutes to load weights; wait until you see
# "Application startup complete." before running Cell 4b-run.
#
# MODEL_HF_ID   = "Qwen/Qwen3-8B"   # or "openai/gpt-oss-20b", "google/gemma-4-4b-it"
# VLLM_FLAGS    = [
#     "--max-model-len", "4096",
#     "--gpu-memory-utilization", "0.90",
#     "--dtype", "auto",
# ]
#
# vllm_proc = subprocess.Popen(
#     [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
#      "--model", MODEL_HF_ID, *VLLM_FLAGS],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
# )
# print(f"vLLM PID: {vllm_proc.pid}  — tail logs at /tmp/vllm.log")
# # Read a few lines to confirm startup progress
# import threading, io
# def _tail(proc, path):
#     with open(path, "wb") as f:
#         for line in proc.stdout:
#             f.write(line)
# threading.Thread(target=_tail, args=(vllm_proc, "/tmp/vllm.log"), daemon=True).start()

print("Setup complete.  When vLLM is ready, run Cell 4b-run.")

In [ ]:
# ── Cell 4b-run: execute the bake-off for one model ──────────────────────────
# Change MODEL_NAME to run a different candidate.  The checkpoint lets you
# resume if the runtime is interrupted.

import pathlib
from prometheus.bakeoff import BakeoffConfig, BakeoffRunner, MODEL_REGISTRY

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME   = "qwen3-8b"   # options: "qwen3-8b", "gpt-oss-20b", "gemma4-4b"
MLFLOW_DIR   = f"{PROMETHEUS_DIR}/mlruns"
CKPT_PATH    = f"{PROMETHEUS_DIR}/bakeoff_{MODEL_NAME}.json"
VLLM_URL     = "http://localhost:8000/v1"

print(f"Model      : {MODEL_NAME}  ({MODEL_REGISTRY[MODEL_NAME].hf_id})")
print(f"MLflow dir : {MLFLOW_DIR}")
print(f"Checkpoint : {CKPT_PATH}")
print()

config = BakeoffConfig(
    model_name=MODEL_NAME,
    n_episodes=200,
    noise=0.05,
    budget=500.0,
    vllm_base_url=VLLM_URL,
    temperature=0.0,
    max_tokens=2048,
    mlflow_tracking_uri=MLFLOW_DIR,
    mlflow_experiment="phase0-bakeoff",
    checkpoint_path=CKPT_PATH,
)

runner = BakeoffRunner(config)

def _progress(episode_idx, result):
    if (episode_idx + 1) % 20 == 0 or result.error:
        tag = f"[ERROR: {result.error[:60]}]" if result.error else ""
        print(f"  Episode {episode_idx+1:4d}/200  "
              f"family={result.family:20s}  acc={result.accuracy:.3f}  "
              f"lat={result.latency_s:.1f}s  {tag}")

summary = runner.run(on_episode=_progress)

print()
print(f"═══ Bake-off complete: {MODEL_NAME} ═══")
print(f"Accuracy       : {summary.accuracy:.3f}  ({summary.n_errors} errors)")
print(f"Mean latency   : {summary.mean_latency_s:.2f}s / episode")
print()
print("Per-law accuracy:")
for law, acc in sorted(summary.per_law_accuracy.items()):
    bar = "█" * int(acc * 20)
    print(f"  {law:20s}  {acc:.3f}  {bar}")
print()
print("Per-family accuracy:")
for fam, acc in sorted(summary.per_family_accuracy.items()):
    bar = "█" * int(acc * 20)
    print(f"  {fam:22s}  {acc:.3f}  {bar}")

In [ ]:
# ── Cell 4b-compare: print the leaderboard across all completed bake-off runs ─
# Run this after completing bake-offs for all candidates.

import json, pathlib

CANDIDATES = ["qwen3-8b", "gpt-oss-20b", "gemma4-4b"]
rows = []

for name in CANDIDATES:
    ckpt = pathlib.Path(f"{PROMETHEUS_DIR}/bakeoff_{name}.json")
    if not ckpt.exists():
        print(f"  {name}: no checkpoint yet")
        continue
    state = json.loads(ckpt.read_text())
    results = list(state["results"].values())
    if not results:
        continue
    n = len(results)
    from prometheus.lab.laws import LAW_NAMES
    total_correct = sum(
        sum(r["claims"].get(k) == r["truth"].get(k) for k in LAW_NAMES)
        for r in results
    )
    acc = total_correct / (n * len(LAW_NAMES))
    n_errors = sum(1 for r in results if r.get("error"))
    lats = [r.get("latency_s", 0) for r in results]
    mean_lat = sum(lats) / len(lats) if lats else 0
    rows.append((name, n, acc, n_errors, mean_lat))

if rows:
    rows.sort(key=lambda r: -r[2])
    print(f"{'Model':<22} {'N':>5} {'Accuracy':>9} {'Errors':>7} {'Lat(s)':>8}")
    print("─" * 56)
    for name, n, acc, nerr, lat in rows:
        winner = " ◀ WINNER" if rows and rows[0][0] == name and len(rows) > 1 else ""
        print(f"{name:<22} {n:>5} {acc:>9.3f} {nerr:>7} {lat:>8.2f}{winner}")
    print()
    if rows:
        winner_name = rows[0][0]
        from prometheus.bakeoff import MODEL_REGISTRY
        spec = MODEL_REGISTRY[winner_name]
        print(f"Selected cortex model: {winner_name}  ({spec.hf_id})")
        print(f"Next step: pin the HF commit hash in models.py → revision field.")
else:
    print("No bake-off results found yet.")

---
## Cell 5 — Propose a variant *(optional)*
Packages the current harness directory as a variant bundle and writes it to
`exchange/proposals/<name>/` for the governor to review.

In Phase 0 the harness is still human-authored; this cell is a dry-run
scaffold for when the Ashby loop starts producing variants.

In [ ]:
import hashlib, json, pathlib, shutil, time

# ── Parameters ────────────────────────────────────────────────────────────────
VARIANT_NAME  = "g1"          # short label, no spaces
HARNESS_DIR   = f"{INSTALL_ROOT}/current/harness"   # source harness to bundle
PROXY_SCORE   = None          # float if you have an evaluator score, else None

# ── Build audit claims from the checkpoint ────────────────────────────────────
cp = pathlib.Path(CHECKPOINT_PATH)
if not cp.exists():
    raise RuntimeError("No checkpoint found — run Cell 4 first.")
cp_state = json.loads(cp.read_text())
audit_claims = {
    seed_str: {k: v for k, v in r["claims"].items()}
    for seed_str, r in cp_state["results"].items()
}

# ── Assemble the bundle ───────────────────────────────────────────────────────
proposal_dir = pathlib.Path(EXCHANGE_DIR) / "proposals" / VARIANT_NAME
if proposal_dir.exists():
    shutil.rmtree(proposal_dir)
proposal_dir.mkdir(parents=True)

# Copy harness files
files_dir = proposal_dir / "files"
if pathlib.Path(HARNESS_DIR).exists():
    shutil.copytree(HARNESS_DIR, files_dir / "harness")
else:
    files_dir.mkdir()
    print(f"Warning: {HARNESS_DIR} not found — bundle will have no files.")

# Write audit claims
(proposal_dir / "audit_claims.json").write_text(
    json.dumps(audit_claims, indent=2, sort_keys=True)
)

# Compute the variant hash (SHA-256 of sorted file contents, matching prometheus.common.variant)
from prometheus.common.variant import load_bundle

# Write a provisional manifest so load_bundle can hash it
manifest = {
    "generation"  : gate.generation + 1,
    "parent_hash" : gate.installed_hash,
    "run_id"      : RUN_ID,
    "created_at"  : time.time(),
}
if PROXY_SCORE is not None:
    manifest["proxy_score"] = PROXY_SCORE
(proposal_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True))

bundle = load_bundle(proposal_dir)
print(f"Bundle name   : {VARIANT_NAME}")
print(f"Variant hash  : {bundle.variant_hash}")
print(f"Generation    : {manifest['generation']}")
print(f"Parent hash   : {gate.installed_hash[:16]}…")
print(f"Audit claims  : {len(audit_claims)} worlds")
print()
print("Bundle written. The governor will review it on the next heartbeat tick.")
print("Check exchange/approvals/ for the decision (allow ~5 minutes for rclone sync).")

---
## Cell 6 — Install an approved variant
After the governor writes an approval to `exchange/approvals/<name>.json`,
call `gate.install()` to atomically swap the harness into place.

In [ ]:
VARIANT_TO_INSTALL = "g1"    # must match Cell 5's VARIANT_NAME

proposal_dir = pathlib.Path(EXCHANGE_DIR) / "proposals" / VARIANT_TO_INSTALL
try:
    dest = gate.install(proposal_dir)
    print(f"Installed     : generation={gate.generation}, dest={dest}")
    print(f"Installed hash: {gate.installed_hash}")
except InstallRefused as e:
    print(f"Install refused: {e}")
    raise
except Halted as e:
    print(f"Halted before install: {e}")
    raise